# 0. Import, device and hyper-parameters

In [1]:
import utils
import torch
import torch.nn as nn
import torch.nn.functional as F
import os


device = utils.try_gpu(0)
print(f"Use the {device} device")

bos_token = 1
eos_token = 2
DATA_PATH = "./data/cmn-eng/cmn.txt"

# 面向完整 cmn-eng（~3.2 万句）；5090 显存充裕
# num_steps=32 可覆盖约 99% 英/中序列（含 bos/eos）
embed_size, num_hiddens, num_layers, dropout = 256, 256, 2, 0.1
batch_size, num_steps = 256, 32
lr, num_epochs = 0.001, 1000



Use the cuda:0 device


# 1. Data loading and pre-processing

In [ ]:
def load_data_TdBSP(data_path):
    """返回 token 序列。源/目标都只含内容 + <eos>，不含 <bos>。
    <bos> 只在训练强制教学 / 推理解码时作为解码器输入。
    """
    with open(data_path, 'r', encoding='utf-8') as f:
        Xs, Ys = [], []
        for line in f:
            line = line.split('\t')
            if len(line) >= 2:
                x = line[0].split(' ')
                x[-1] = x[-1][:-1]          # 去掉英文句末标点
                x.append('<eos>')
                Xs.append(x)

                y = list(line[1])
                y = y[:-1]                  # 去掉中文句末标点
                y.append('<eos>')
                Ys.append(y)
        return Xs, Ys


In [ ]:
def preprocess_data(Xs, Ys):
    if isinstance(Xs[0], str):
        Xs = utils.tokenize(Xs, token='word')
        Ys = utils.tokenize(Ys, token='word')
    X_vocab = utils.Vocab(Xs, reserved_tokens=['<bos>', '<eos>'])
    Y_vocab = utils.Vocab(Ys, reserved_tokens=['<bos>', '<eos>'])
    X = [[X_vocab[tk] for tk in line] for line in Xs]
    Y = [[Y_vocab[tk] for tk in line] for line in Ys]
    return X, Y, X_vocab, Y_vocab


In [ ]:
def data_loader_TdBSP(X, Y, batch_size, num_steps, pad=0):
    """将索引序列 pad/截断后返回 DataLoader。
    每个 batch: (X, X_valid_len, Y, Y_valid_len)。
    pad 默认 0，对应 Vocab 的 <unk>；若词表有 <pad> 请传入其索引。
    """
    def truncate_pad(line):
        if len(line) > num_steps:
            return line[:num_steps]
        return line + [pad] * (num_steps - len(line))

    def build_array(lines):
        array = torch.tensor([truncate_pad(l) for l in lines])
        valid_len = (array != pad).type(torch.int32).sum(1)
        return array, valid_len

    X_array, X_valid_len = build_array(X)
    Y_array, Y_valid_len = build_array(Y)
    dataset = torch.utils.data.TensorDataset(
        X_array, X_valid_len, Y_array, Y_valid_len)
    return torch.utils.data.DataLoader(dataset, batch_size, shuffle=True)


In [ ]:
Xs, Ys = load_data_TdBSP(DATA_PATH)
X, Y, X_vocab, Y_vocab = preprocess_data(Xs, Ys)

print(f"The first 5 words in the source sequence: \n{Xs[:5]}")
print(f"The first 5 words in the target sequence: \n{Ys[:5]}")
print(f"The vocabulary size of source sequence: {len(X_vocab)}, The first 10 words in the vocabulary: \n{X_vocab.idx_to_token[:10]}")
print(f"The vocabulary size of target sequence: {len(Y_vocab)}, The first 10 words in the vocabulary: \n{Y_vocab.idx_to_token[:10]}")

train_iter = data_loader_TdBSP(X, Y, batch_size, num_steps)


# 2. Model and function

In [4]:
class Seq2SeqEncoder(nn.Module):
    """用于序列到序列学习的循环神经网络编码器。"""
    def __init__(self, vocab_size, embed_size, num_hiddens, num_layers,
                 dropout=0, **kwargs):
        super(Seq2SeqEncoder, self).__init__(**kwargs)
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.rnn = nn.GRU(embed_size, num_hiddens, num_layers, dropout=dropout)
    
    def forward(self, X, *args):
        # Input size: (batch_size, seq_len)
        X = self.embedding(X)  # (batch_size, seq_len, embed_size)
        X = X.permute(1, 0, 2)  # (seq_len, batch_size, embed_size)
        output, state = self.rnn(X)
        # output size: (seq_len, batch_size, num_hiddens)
        # state size: (num_layers, batch_size, num_hiddens)
        return output, state

In [ ]:
class Seq2SeqDecoder(nn.Module):
    """用于序列到序列学习的循环神经网络解码器。"""
    def __init__(self, vocab_size, embed_size, num_hiddens, num_layers,
                 dropout=0, **kwargs):
        super(Seq2SeqDecoder, self).__init__(**kwargs)
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.rnn = nn.GRU(embed_size + num_hiddens, num_hiddens, num_layers,
                          dropout=dropout)
        self.dense = nn.Linear(num_hiddens, vocab_size)

    def init_state(self, enc_outputs, *args):
        return enc_outputs[1]

    def forward(self, X, state):
        # Input size: (batch_size, seq_len)
        X = self.embedding(X)  # (batch_size, seq_len, embed_size)
        X = X.permute(1, 0, 2)  # (seq_len, batch_size, embed_size)
        context = state[-1].repeat(X.shape[0], 1, 1)
        X_and_context = torch.cat((X, context), dim=2)
        output, state = self.rnn(X_and_context, state)
        output = self.dense(output).permute(1, 0, 2)
        return output, state


In [ ]:
def seq_mask(X, valid_len, value=0):
    X = X.clone()
    maxlen = X.size(1)
    mask = torch.arange((maxlen), dtype=torch.float32,
                        device=X.device)[None, :] < valid_len[:, None]
    X[~mask] = value
    return X


In [ ]:
class MaskedSoftmaxCELoss(nn.CrossEntropyLoss):
    """带遮蔽的softmax交叉熵损失函数。"""
    def forward(self, pred, label, valid_len):
        weights = torch.ones_like(label)
        weights = seq_mask(weights, valid_len)
        self.reduction = 'none'
        unweighted_loss = super(MaskedSoftmaxCELoss, self).forward(
            pred.permute(0, 2, 1), label)
        weighted_loss = (unweighted_loss * weights).mean(dim=1)
        return weighted_loss


In [ ]:
def train_seq2seq(net, data_iter, lr, num_epochs, tgt_vocab, device):
    """训练序列到序列模型"""
    def xavier_init_weights(m):
        if type(m) == nn.Linear:
            nn.init.xavier_uniform_(m.weight)
        if type(m) == nn.GRU:
            for param in m._flat_weights_names:
                if "weight" in param:
                    nn.init.xavier_uniform_(m._parameters[param])

    net.apply(xavier_init_weights)
    net.to(device)
    optimizer = torch.optim.Adam(net.parameters(), lr=lr)
    loss = MaskedSoftmaxCELoss()
    net.train()
    animator = utils.AnimatorSimple(xlabel='epoch', ylabel='loss',
                                    xlim=[10, num_epochs])
    for epoch in range(num_epochs):
        timer = utils.Timer()
        metric = utils.Accumulator(2)  # 训练损失总和，词元数量
        for batch in data_iter:
            optimizer.zero_grad()
            X, X_valid_len, Y, Y_valid_len = [x.to(device) for x in batch]
            bos = torch.tensor([tgt_vocab['<bos>']] * Y.shape[0],
                               device=device).reshape(-1, 1)
            dec_input = torch.cat([bos, Y[:, :-1]], 1)  # 强制教学
            Y_hat, _ = net(X, dec_input, X_valid_len)
            l = loss(Y_hat, Y, Y_valid_len)
            l.sum().backward()
            utils.grad_clipping(net, 1)
            num_tokens = Y_valid_len.sum()
            optimizer.step()
            with torch.no_grad():
                metric.add(l.sum(), num_tokens)
        if (epoch + 1) % 10 == 0:
            animator.add(epoch + 1, (metric[0] / metric[1],))
    print(f'loss {metric[0] / metric[1]:.3f}, {metric[1] / timer.stop():.1f} '
          f'tokens/sec on {str(device)}')


In [ ]:
class EncoderDecoder(nn.Module):
    """用于将编码器和解码器组合在一起的模型。"""
    def __init__(self, encoder, decoder):
        super(EncoderDecoder, self).__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, enc_X, dec_X, X_valid_len):
        enc_outputs = self.encoder(enc_X, X_valid_len)
        dec_state = self.decoder.init_state(enc_outputs, X_valid_len)
        return self.decoder(dec_X, dec_state)


In [ ]:
encoder = Seq2SeqEncoder(len(X_vocab), embed_size, num_hiddens, num_layers,
                         dropout)
decoder = Seq2SeqDecoder(len(Y_vocab), embed_size, num_hiddens, num_layers,
                         dropout)
net = EncoderDecoder(encoder, decoder)


# 3. Training and generation


In [ ]:
train_seq2seq(net, train_iter, lr, num_epochs, Y_vocab, device)


In [ ]:
def translate(net, src_sentence, src_vocab, tgt_vocab, num_steps, device):
    """贪心解码：将英文源句翻译为目标序列（中文按字拼接）。"""
    net.eval()
    # 与 load_data_TdBSP 一致：去句末标点 + <eos>（源序列不含 <bos>）
    src_tokens = src_sentence.strip().split(' ')
    if src_tokens:
        src_tokens[-1] = src_tokens[-1][:-1]
    src_tokens = src_tokens + ['<eos>']
    src_ids = src_vocab[src_tokens]
    enc_valid_len = torch.tensor([len(src_ids)], device=device)

    pad = 0  # 与 data_loader_TdBSP 默认 pad 一致（Vocab 的 <unk>）
    if len(src_ids) > num_steps:
        src_ids = src_ids[:num_steps]
    else:
        src_ids = src_ids + [pad] * (num_steps - len(src_ids))

    enc_X = torch.tensor(src_ids, dtype=torch.long, device=device).unsqueeze(0)
    enc_outputs = net.encoder(enc_X, enc_valid_len)
    dec_state = net.decoder.init_state(enc_outputs, enc_valid_len)
    # <bos> 只作为解码器第一步输入
    dec_X = torch.tensor([[tgt_vocab['<bos>']]], dtype=torch.long, device=device)

    output_seq = []
    for _ in range(num_steps):
        Y, dec_state = net.decoder(dec_X, dec_state)
        dec_X = Y.argmax(dim=2)  # 下一步输入 = 当前预测
        pred = int(dec_X.squeeze(dim=0).item())
        if pred == tgt_vocab['<eos>']:
            break
        output_seq.append(pred)
    return ''.join(tgt_vocab.to_tokens(output_seq))


In [ ]:
print(translate(net, "Hi.", X_vocab, Y_vocab, num_steps, device))
print(translate(net, "I won!", X_vocab, Y_vocab, num_steps, device))
print(translate(net, "Hello!", X_vocab, Y_vocab, num_steps, device))
